**ISD-1020 · Master ISD (apprenticeship) · Université Paris-Saclay**

**Before any edit:** `File → Save a copy in Drive`.
Work only on **your** copy.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch

print("sklearn", sklearn.__version__, "| torch", torch.__version__)

# A pretrained ResNet18 — these are not *your* labels

ISD-1020 · Monday 28 September 2026 · afternoon (Stéphane Rivaud) ·
PUIO B210. **Demonstration ~20 min**, watched together — not a lab to
fill during the session.

Thread of the module: **1. Problem · 2. Data · 3. Baseline · 4. Model · 5. Ablation ·
6. Error analysis → Recommendation**.
Here: a network already trained on ImageNet (web photos, 1,000 English
classes from 2012), run on **six images from the repository** (signs,
car, truck, bus, train). CPU only.

Honest arrival point: ImageNet **does not have** European highway-code
signs. The top-5 will sometimes be right on a vehicle, often
comic on a STOP. That is the point — not a model failure.

## The images — ⏱ ~3 min

Folder `images/` next to this notebook, or in the repository.
Licences: `../images/ATTRIBUTION.md`.

In [ ]:
import os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

from PIL import Image

IMAGES = [
    ("french_stop.jpg", "STOP / yield / parking signs (Lyon)"),
    ("yield_sign.jpg", "Yield sign (New Hampshire)"),
    ("peugeot_207.jpg", "Peugeot 207 (Katowice)"),
    ("truck_trailer.jpg", "Heavy goods vehicle with trailer"),
    ("paris_bus.jpg", "Tourist bus, Louvre"),
    ("passenger_train.jpg", "Passenger train (diesel locomotive)"),
]
_DEMO_IMAGE_TIMEOUT = 5  # seconds; fail fast if raw GitHub does not answer
_DEMO_IMAGES_MISSING = (
    "Demonstration images not found. They live in the "
    "images/ folder the instructor shared, next to the notebook. "
    "Place the six JPEGs there, or follow the demonstration on the projector."
)


def _local_demo_images() -> Path | None:
    """Images next to this notebook, or under the course repository."""
    for start in (Path.cwd(), *Path.cwd().parents):
        for relative in (
            Path("images"),
            Path("sessions") / "d2-2026-09-28" / "pm" / "images",
        ):
            path = start / relative
            if (path / "french_stop.jpg").is_file():
                return path
    return None


def _demo_image_cache_dir() -> Path:
    content = Path("/content")
    if content.is_dir() and os.access(content, os.W_OK):
        dest = content / "isd-1020-cache" / "demo-images"
    else:
        dest = Path("/tmp/isd-1020-cache") / "demo-images"
    dest.mkdir(parents=True, exist_ok=True)
    return dest


def _github_raw_enabled() -> bool:
    flag = os.environ.get("ISD1020_USE_GITHUB_RELEASE", "").strip().lower()
    return flag not in {"0", "false", "no", "off"}


def _fetch_demo_images() -> Path | None:
    if not _github_raw_enabled():
        return None
    slug = os.environ.get("ISD1020_REPO", "").strip() or "stephane-rivaud/M2-ISD-ML-DL"
    dest = _demo_image_cache_dir()
    base = (
        f"https://raw.githubusercontent.com/{slug}/main/"
        "sessions/d2-2026-09-28/pm/images"
    )
    try:
        for name, _caption in IMAGES:
            target = dest / name
            if target.is_file():
                continue
            print(f"images: downloading {name}", flush=True)
            request = Request(
                f"{base}/{name}",
                headers={"User-Agent": "ISD-1020-course-pack/0.1"},
            )
            with urlopen(request, timeout=_DEMO_IMAGE_TIMEOUT) as response:
                target.write_bytes(response.read())
    except (OSError, URLError) as exc:
        print(
            f"images: download unavailable ({type(exc).__name__}), "
            "local folder required",
            flush=True,
        )
        return None
    if any(not (dest / name).is_file() for name, _caption in IMAGES):
        return None
    return dest


def _find_demo_images() -> Path:
    local = _local_demo_images()
    if local is not None:
        return local
    fetched = _fetch_demo_images()
    if fetched is not None:
        return fetched
    print(_DEMO_IMAGES_MISSING, flush=True)
    raise SystemExit(_DEMO_IMAGES_MISSING)


IMAGE_DIR = _find_demo_images()
missing = [name for name, _caption in IMAGES if not (IMAGE_DIR / name).is_file()]
if missing:
    print(_DEMO_IMAGES_MISSING, flush=True)
    raise SystemExit(_DEMO_IMAGES_MISSING)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, (name, caption) in zip(axes.ravel(), IMAGES):
    ax.imshow(Image.open(IMAGE_DIR / name))
    ax.set_title(caption, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("6 files:", IMAGE_DIR)

## torchvision and ResNet18 — ⏱ ~4 min

Colab ships `torchvision` next to `torch`. The repository `.venv`
installs it via `pyproject.toml`. No install in this cell.

**Next**: ResNet18 ImageNet weights weigh **~45 MB**,
downloaded on the first call. That is not a crash. Without a network,
the cell must fail with a readable message, not stay silent.

In [ ]:
# Colab ships torchvision next to torch; the course .venv lists it in pyproject.toml.
try:
    import torchvision
except ImportError:
    raise RuntimeError(
        "torchvision is missing. Colab ships it with torch: "
        "Runtime → Restart session, then rerun. "
        "Locally, install the repository dependencies. "
        "This demonstration is followed on the projector."
    ) from None

print("torchvision", torchvision.__version__)

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18

print("Downloading ResNet18 weights (~45 MB, once).")
print("If nothing prints for more than a minute: check the network,")
print("this is not a silent kernel freeze.")
try:
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
except Exception as exc:
    raise RuntimeError(
        "Could not download ResNet18 weights (~45 MB). "
        "Check the connection, then rerun this cell."
    ) from exc

model.eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]
print("ImageNet classes:", len(categories))
print("contains 'street sign':", "street sign" in categories)

## Top-5 per image — ⏱ ~5 min

Preprocessing = `weights.transforms()` (the one that goes with *these*
weights). Labels = `weights.meta["categories"]`, not a copied
list. `model.eval()`: no dropout, no batch-norm in train
mode.

In [ ]:
@torch.no_grad()
def top5(path):
    """Return five (class_name, probability) pairs for a RGB JPEG/PNG."""
    image = Image.open(path).convert("RGB")
    batch = preprocess(image).unsqueeze(0)
    probs = model(batch).squeeze(0).softmax(0)
    values, indices = torch.topk(probs, 5)
    return [(categories[int(i)], float(v)) for v, i in zip(values, indices)]


for name, caption in IMAGES:
    print(f"\n{name} — {caption}")
    for rank, (label, prob) in enumerate(top5(IMAGE_DIR / name), start=1):
        print(f"  {rank}. {label:30s} {100 * prob:5.1f} %")

ImageNet knows `street sign`, `traffic light`, `sports car`,
`trailer truck`, `passenger car`, `school bus`, `electric locomotive`.
Not `STOP`, not "give way", not a 50 speed limit. A
pretrained backbone brings **general features**, not the
vocabulary of your job.

**Second limit, visible on `truck_trailer.jpg`.** Scores there are
all low and scattered (`limousine`, `cab`, `ambulance`…), whereas the
207 in profile comes out at 87%. The HGV photo is a **scene**
from above: a truck, six cars, buildings, bikes. Yet this
network assigns **one label to the whole image**. A sharp subject that
fills the frame gives a frank score; a cluttered scene gives five
lukewarm hypotheses. That is exactly what happens to a camera placed
on a real site, and that is why object detection (one box
per object) is another problem than classification.

**One sentence of fine-tuning.** We replace the head `Linear(512, 1000)`
with `Linear(512, n_job_classes)` and first retrain that
head (then, optionally, we unfreeze the backbone).

## Exercise — ⏱ ~3 min

Among the six top-5s, **which** is the most off relative to what
the photo shows? Tie that to the ImageNet vocabulary (1,000
classes, 2012): the network does not have your sign labels. Write
two sentences, then what fine-tuning changes in the **head**.

In [ ]:
# To complete
...

## Fast track — ⏱ ~2 min (optional)

ImageNet classes whose name evokes a vehicle or a sign. Useful
to see *what the network is allowed to say*.

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
keys = (
    "car", "truck", "bus", "train", "locomotive",
    "street sign", "traffic light", "minibus", "trolleybus",
)
for index, label in enumerate(categories):
    lowered = label.lower()
    if any(key in lowered for key in keys):
        print(f"{index:4d}  {label}")

## Attributions

- **torchvision / PyTorch** : Meta (Facebook) PyTorch,
  [BSD-3-Clause](https://github.com/pytorch/vision/blob/main/LICENSE).
- **ResNet** : He, K. ; Zhang, X. ; Ren, S. ; Sun, J. *Deep Residual
  Learning for Image Recognition*, CVPR 2016,
  [arXiv:1512.03385](https://arxiv.org/abs/1512.03385).
- **Weights** `ResNet18_Weights.IMAGENET1K_V1`: ImageNet-1k classification
  via torchvision (download at runtime, not in git).
- **Six photographs**: CC0 1.0, detail in
  [`../images/ATTRIBUTION.md`](../images/ATTRIBUTION.md).
- **Keras sheet** (companies on TF, not the module):
  [`common/keras-equivalence.md`](../../../../common/keras-equivalence.md).

## To go further at home

1. PyTorch tutorial *Transfer learning for computer vision*:
   [transfer_learning_tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
2. Module Keras equivalence sheet:
   [`common/keras-equivalence.md`](../../../../common/keras-equivalence.md)

## Empirical protocol — outside the demo clock

The six cells below are the thread of the module (and of the oral exam).
Answer in prose, tied to *this* demo.

## Problem

Answer in prose (not only code).

1. Which **business decision** should the model inform, and what is the **target**?
2. Which **metric** matches that decision, and why not accuracy alone?
3. How do you separate training and test **without leakage** (split unit, time, stratification)?

## Data

Answer in prose (not only code).

1. What are the variables, their types, and any notable **imbalances** or volumes?
2. Which columns must you **not** use (identifiers, leakages, target sub-flags)?
3. What do you know about **quality** (missing values, outliers, drift) and temporal or business coverage?

## Baseline

Answer in prose (not only code).

1. Which **naive baseline** (majority class, mean, seasonal) and what score does it get?
2. Why is this baseline the **honest floor**, and not a "deliberately weak" model?
3. Which score must you **beat** to justify a more complex model?

## Model

Answer in prose (not only code).

1. Which **model family** do you choose, and why (not "because it is deep")?
2. How do you train it (split, validation, hyperparameters, loss)?
3. Does the model **beat the baseline** on the chosen metric, on an uncontaminated split?

## Ablation

Answer in prose (not only code).

1. What happens if you remove a family of variables, a regulariser, or a block of the network?
2. Which choice (features, architecture, horizon, threshold) **actually changes** the score?
3. Does the gain justify the extra **complexity** relative to the baseline?

## Error analysis → Recommendation

Answer in prose (not only code).

1. Where does the model go wrong (**segments**, error types, horizon)?
2. Are these errors **costly** for the business, and what would you change in the data or the model?
3. What concrete **recommendation**: deploy, do not deploy, stay on the baseline, or collect this specific data?